In [15]:
import pandas as pd
from pycaret.classification import load_model
from sklearn.metrics import log_loss, f1_score
import mlflow

# Carregar o modelo salvo pelo PyCaret
model = load_model("../Model/artifacts/modelo_final")

# Carregar os dados de produção
df_prod = pd.read_parquet("../../Data/Raw/dataset_kobe_prod.parquet")

# Imprimir as colunas disponíveis (para auxiliar na escolha dos recursos)
print("Colunas disponíveis em df_prod:")
print(df_prod.columns)

# Defina as colunas que o modelo espera como entrada.
cols = ["lat", "lon", "minutes_remaining", "period", "playoffs", "shot_distance"]

# Realiza as predições
predictions = model.predict(df_prod[cols])

# Se o modelo oferecer probabilidades, obtenha-as para cálculo do log_loss
if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(df_prod[cols])
else:
    probabilities = None

# Salvar os resultados com as predições
df_prod["prediction"] = predictions
df_prod.to_parquet("../../Data/Processed/predictions_prod.parquet")

# Registrar métricas (caso a coluna alvo esteja presente e contenha dados válidos)
if "shot_made_flag" in df_prod.columns:
    # Filtrar linhas onde shot_made_flag não é NaN
    valid_idx = df_prod["shot_made_flag"].notna()
    if valid_idx.sum() > 0:
        metrics = {}
        y_true = df_prod.loc[valid_idx, "shot_made_flag"]
        y_pred = predictions[valid_idx]
        if probabilities is not None:
            prob_valid = probabilities[valid_idx]
            loss = log_loss(y_true, prob_valid)
            metrics["log_loss_prod"] = loss
        f1 = f1_score(y_true, y_pred)
        metrics["f1_prod"] = f1
        
        with mlflow.start_run(run_name="PrevisaoProducao"):
            mlflow.log_metrics(metrics)
    else:
        print("Nenhuma linha com 'shot_made_flag' válido para calcular métricas.")


Transformation Pipeline and Model Successfully Loaded
Colunas disponíveis em df_prod:
Index(['action_type', 'combined_shot_type', 'game_event_id', 'game_id', 'lat',
       'loc_x', 'loc_y', 'lon', 'minutes_remaining', 'period', 'playoffs',
       'season', 'seconds_remaining', 'shot_distance', 'shot_made_flag',
       'shot_type', 'shot_zone_area', 'shot_zone_basic', 'shot_zone_range',
       'team_id', 'team_name', 'game_date', 'matchup', 'opponent', 'shot_id'],
      dtype='object')
